LeNet由两个部分组成，卷积编码器和全连接层密集块

In [8]:
import torch
from torch import nn
from d2l import torch as d2l

class Reshape(nn.Module):
    def forward(self, x):
        return x.view(-1,1,28,28)
#把张量换一种形状，不动数据

net=torch.nn.Sequential(
    Reshape(),nn.Conv2d(1,6,kernel_size=5,padding=2),nn.Sigmoid(),
    nn.AvgPool2d(2,stride=2),
    nn.Conv2d(6,16,kernel_size=5),nn.Sigmoid(),
    nn.AvgPool2d(2,stride=2),nn.Flatten(),
    nn.Linear(16*5*5,120),nn.Sigmoid(),
    nn.Linear(120,84),nn.Sigmoid(),
    nn.Linear(84,10)
)

好复杂的层，怎么这么多！

直观感受这每一层的输入和输出

In [9]:
X=torch.rand(size=(1,1,28,28),dtype=torch.float32)
for layer in net:
    X=layer(X)
    print(layer.__class__.__name__,'output shape:\t',X.shape)

Reshape output shape:	 torch.Size([1, 1, 28, 28])
Conv2d output shape:	 torch.Size([1, 6, 28, 28])
Sigmoid output shape:	 torch.Size([1, 6, 28, 28])
AvgPool2d output shape:	 torch.Size([1, 6, 14, 14])
Conv2d output shape:	 torch.Size([1, 16, 10, 10])
Sigmoid output shape:	 torch.Size([1, 16, 10, 10])
AvgPool2d output shape:	 torch.Size([1, 16, 5, 5])
Flatten output shape:	 torch.Size([1, 400])
Linear output shape:	 torch.Size([1, 120])
Sigmoid output shape:	 torch.Size([1, 120])
Linear output shape:	 torch.Size([1, 84])
Sigmoid output shape:	 torch.Size([1, 84])
Linear output shape:	 torch.Size([1, 10])


思想目的：减少高宽，增加通道。然后全连接层来实现。

In [10]:
batch_size=256
train_iter,test_iter=d2l.load_data_fashion_mnist(batch_size=batch_size)

In [11]:
def evaluate_accuracy_gpu(net,train_iter,device=None):
    "使用GPU计算模型在数据集上的精度"
    if isinstance(net,torch.nn.Module):
        net.eval()
        if not device:
            device=next(net.parameters()).device
    metric=d2l.Accumulator(2)
    for X,y in train_iter:
        if isinstance(X,list):
            X=[x.to(device) for x in X]
        else:
            X=X.to(device)
        y=y.to(device)
        metric.add(d2l.accuracy(net(X),y),y.numel())
    return metric[0]/metric[1]

## 训练函数 train_ch6（教材 6.6 标准版，含中文注释）

每次训练套路都一样：初始化权重 → 搬到 GPU → 定义优化器和损失 → 逐 batch
前向/反向/更新 → 累加统计 → 画曲线。封装好以后换网络也能直接用。

In [12]:
def train_ch6(net, train_iter, test_iter, num_epochs, lr, device):
    """用 GPU 训练模型（d2l 第 6 章标准版）"""
    # ---------- 1. 权重初始化 ----------
    # 对网络里每个 Linear / Conv2d 层，用 Xavier 均匀分布初始化权重
    # （Sigmoid 激活配 Xavier，能缓解梯度消失，呼应第 21 节）
    def init_weights(m):
        if type(m) == nn.Linear or type(m) == nn.Conv2d:
            nn.init.xavier_uniform_(m.weight)
    net.apply(init_weights)   # apply 会递归遍历每个子模块并执行该函数

    # ---------- 2. 搬到 GPU + 准备优化器和损失 ----------
    print('training on', device)
    net.to(device)                              # 模型参数搬到 device
    optimizer = torch.optim.SGD(net.parameters(), lr=lr)  # 随机梯度下降
    loss = nn.CrossEntropyLoss()                # 交叉熵（内置 softmax，输入直接给 logits）

    # ---------- 3. 动画绘图器：实时画三条曲线 ----------
    animator = d2l.Animator(xlabel='epoch', xlim=[1, num_epochs],
                            legend=['train loss', 'train acc', 'test acc'])
    timer, num_batches = d2l.Timer(), len(train_iter)

    # ---------- 4. 主循环 ----------
    for epoch in range(num_epochs):
        # 累加器 3 个槽位 = [损失总和, 预测正确数, 样本总数]
        metric = d2l.Accumulator(3)
        net.train()   # 开训练模式（本例无 Dropout/BN，写上是养成习惯）

        for i, (X, y) in enumerate(train_iter):
            timer.start()
            optimizer.zero_grad()               # ① 清空上一步残留的梯度
            X, y = X.to(device), y.to(device)   # ② 数据搬到同一设备
            y_hat = net(X)                      # ③ 前向传播
            l = loss(y_hat, y)                  # ④ 计算损失
            l.backward()                        # ⑤ 反向传播求梯度
            optimizer.step()                    # ⑥ 按梯度更新参数

            with torch.no_grad():               # 统计指标不需要梯度，不建计算图
                metric.add(l * X.shape[0],      # 该 batch 的损失总和（×样本数才能平均）
                           d2l.accuracy(y_hat, y),  # 该 batch 预测正确的个数
                           X.shape[0])           # 该 batch 样本数
            timer.stop()
            train_l = metric[0] / metric[2]     # 到目前为止的平均损失
            train_acc = metric[1] / metric[2]   # 到目前为止的训练精度
            # 每个 epoch 画 5 次中间结果，曲线更平滑
            if (i + 1) % (num_batches // 5) == 0 or i == num_batches - 1:
                animator.add(epoch + (i + 1) / num_batches,
                             (train_l, train_acc, None))

        # 每个 epoch 结束：在测试集上评估精度（调用上面自己写的函数）
        test_acc = evaluate_accuracy_gpu(net, test_iter)
        animator.add(epoch + 1, (None, None, test_acc))

    # ---------- 5. 收尾报告 ----------
    print(f'loss {train_l:.3f}, train acc {train_acc:.3f}, '
          f'test acc {test_acc:.3f}')
    print(f'{metric[2] * num_epochs / timer.sum():.1f} examples/sec '
          f'on {str(device)}')

In [ ]:
# ---------- 训练 LeNet ----------
# lr=0.9：Sigmoid 网络（无 BN）需要较大学习率；10 个 epoch 足够让精度 >98%
lr, num_epochs = 0.9, 100
train_ch6(net, train_iter, test_iter, num_epochs, lr, d2l.try_gpu())